# Train Multi-Scale Crop Health from Full Zip Upload on Kaggle

This notebook trains your ablation experiments securely using the Kaggle environment, assuming you uploaded your entire 6-7GB locally zipped project folder as a Kaggle Dataset.

**Note on Kaggle's File System:**
Kaggle mounts uploaded datasets to `/kaggle/input` and strictly limits them to **read-only**. However, your training script needs write-access to save log outputs, checkpoints, and metrics. 
To bypass this without wasting 7GB of disk space and 10 minutes copying your raw data, this notebook smartly extracts only the scripts and small files into your writable zone (`/kaggle/working`), then cleanly symlinks your raw dataset into place!

In [ ]:
import os, glob, shutil

print("Available input datasets:", os.listdir('/kaggle/input'))

# Kaggle datasets can randomly nest folders if they were zipped as a containing directory.
# Search dynamically for the true codebase root by finding 'experiments/run_experiment.py'
true_root = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'experiments' in dirs and 'data' in dirs and 'run_experiment.py' in os.listdir(os.path.join(root, 'experiments')):
        true_root = root
        break
        
if not true_root:
    raise RuntimeError("Could not find the 'experiments' and 'data' folders in /kaggle/input! Is the zip missing them?")

print('Found True Source Path:', true_root)

WORKING_PROJECT_PATH = '/kaggle/working/multi-scale-crop-health'
if os.path.exists(WORKING_PROJECT_PATH):
    shutil.rmtree(WORKING_PROJECT_PATH)
os.makedirs(WORKING_PROJECT_PATH, exist_ok=True)


In [ ]:
# 1. Copy only the codebase files to the working directory (Skip heavy 'data' & 'venv' etc.)
print('Copying scripts to writable zone...')
for item in os.listdir(true_root):
    if item in ['data', 'venv_cuda', '.git', '__pycache__']:
        continue  # Skip these
        
    s = os.path.join(true_root, item)
    d = os.path.join(WORKING_PROJECT_PATH, item)
    
    if os.path.isdir(s):
        shutil.copytree(s, d)
    else:
        shutil.copy2(s, d)
        
print('Codebase ready in', WORKING_PROJECT_PATH)

In [ ]:
# 2. Symlink the huge 'data' folder directly into the codebase 
SOURCE_DATA = os.path.join(true_root, 'data')
DEST_DATA = os.path.join(WORKING_PROJECT_PATH, 'data')

if os.path.exists(SOURCE_DATA):
    os.symlink(SOURCE_DATA, DEST_DATA)
    print(f"Successfully linked {SOURCE_DATA} -> {DEST_DATA}")
else:
    print(f"WARNING: 'data' folder not found inside {true_root}. Are you sure your zip hierarchy is correct?")

In [ ]:
# Switch terminal environment to codebase
%cd /kaggle/working/multi-scale-crop-health

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

# Dependencies are generally built-in, but enforce any missing ones
!pip -q install tifffile

In [ ]:
# Launch training ablation runner
# Need smaller batch sizes on Kaggle T4x2 to prevent OOM errors.
!python experiments/run_experiment.py --ablation

In [ ]:
# Outputs are saved locally in /kaggle/working/multi-scale-crop-health/outputs
# Use the Output sidebar to retrieve your JSONs, models, and metric graphs.